In [1]:
import numpy as np
import pandas as pd
import requests
import io

In [6]:
# header=1 옵션으로 맨 위의 '실시간 측정망 현황' 줄을 무시하고 불러옵니다.
files = ['data\Jungryangchun.xls', 'data\Ahnyangchun.xls', 'data\Tanchun.xls']
target_columns = ['날짜', '시간', '측정소명', '수온', 'pH', '용존산소(㎎/L)', '총질소(㎎/L)', '총인(㎎/L)', '총유기탄소']
# 평균을 낼 수치형 컬럼만 따로 지정
numeric_columns = ['수온', 'pH', '용존산소(㎎/L)', '총질소(㎎/L)', '총인(㎎/L)', '총유기탄소']

processed_list = []

for f in files:
    # [단계 1] HTML 형식 읽기
    tables = pd.read_html(f, encoding='utf-8')
    df = tables[0]
    
    # [단계 2] 필요한 컬럼만 선택
    df = df[target_columns]
    
    # [단계 3] 날짜 형식 변환 및 5~10월 필터링
    df['날짜'] = pd.to_datetime(df['날짜'].astype(str))
    df = df[df['날짜'].dt.month.between(5, 10)]
    
    # [단계 4] ★수정 포인트: 15시 필터링 대신 '일별 평균' 계산★
    # 각 하천 내에서 시간대별로 흩어진 데이터를 하루치 평균으로 압축합니다.
    df_daily = df.groupby('날짜')[numeric_columns].mean(numeric_only=True).reset_index()
    
    print(df_daily.info())
    
    processed_list.append(df_daily)

# [단계 5] 3개 하천의 일별 평균 데이터들을 하나로 통합
river_concat = pd.concat(processed_list, ignore_index=True)

# [단계 6] 날짜별로 다시 그룹화하여 3개 하천 사이의 최종 평균값 계산
# (중랑천, 안양천, 탄천의 그날 평균들을 다시 평균 내어 '서울 대표 수질' 생성)
final_river_df = river_concat.groupby('날짜').mean(numeric_only=True).reset_index()

# [단계 7] 병합용 ID('tm') 생성 (YYYYMMDD 형식)
final_river_df['tm'] = final_river_df['날짜'].dt.strftime('%Y%m%d')

# 최종 결과 확인
print(f"최종 데이터 개수: {len(final_river_df)}행 (기대치: 1093)")
display(final_river_df.head())

C:\Users\chlal\AppData\Local\Temp\ipykernel_23004\5129928.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['날짜'] = pd.to_datetime(df['날짜'].astype(str))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1093 entries, 0 to 1092
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   날짜         1093 non-null   datetime64[ns]
 1   수온         1064 non-null   float64       
 2   pH         1064 non-null   float64       
 3   용존산소(㎎/L)  1064 non-null   float64       
 4   총질소(㎎/L)   1064 non-null   float64       
 5   총인(㎎/L)    1063 non-null   float64       
 6   총유기탄소      780 non-null    float64       
dtypes: datetime64[ns](1), float64(6)
memory usage: 59.9 KB
None


C:\Users\chlal\AppData\Local\Temp\ipykernel_23004\5129928.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['날짜'] = pd.to_datetime(df['날짜'].astype(str))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1093 entries, 0 to 1092
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   날짜         1093 non-null   datetime64[ns]
 1   수온         1039 non-null   float64       
 2   pH         1039 non-null   float64       
 3   용존산소(㎎/L)  1039 non-null   float64       
 4   총질소(㎎/L)   1031 non-null   float64       
 5   총인(㎎/L)    1040 non-null   float64       
 6   총유기탄소      1035 non-null   float64       
dtypes: datetime64[ns](1), float64(6)
memory usage: 59.9 KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1093 entries, 0 to 1092
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   날짜         1093 non-null   datetime64[ns]
 1   수온         1037 non-null   float64       
 2   pH         1034 non-null   float64       
 3   용존산소(㎎/L)  1036 non-null   float64       
 4   총질소(㎎/L)   10

C:\Users\chlal\AppData\Local\Temp\ipykernel_23004\5129928.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['날짜'] = pd.to_datetime(df['날짜'].astype(str))


,날짜,수온,pH,용존산소(㎎/L),총질소(㎎/L),총인(㎎/L),총유기탄소,tm
0,2020-05-01,21.298611,7.298611,6.229167,8.086181,0.116528,6.862500,20200501
1,2020-05-02,21.418056,7.183333,5.615278,7.697153,0.201667,5.084722,20200502
2,2020-05-03,22.005556,7.166667,5.736111,7.444375,0.200139,5.004167,20200503
3,2020-05-04,23.216667,7.169444,5.648611,7.402917,0.144861,5.172348,20200504
4,2020-05-05,20.429167,7.102778,5.350000,7.886250,0.209861,5.837500,20200505


In [7]:
final_river_df.head()

,날짜,수온,pH,용존산소(㎎/L),총질소(㎎/L),총인(㎎/L),총유기탄소,tm
0,2020-05-01,21.298611,7.298611,6.229167,8.086181,0.116528,6.862500,20200501
1,2020-05-02,21.418056,7.183333,5.615278,7.697153,0.201667,5.084722,20200502
2,2020-05-03,22.005556,7.166667,5.736111,7.444375,0.200139,5.004167,20200503
3,2020-05-04,23.216667,7.169444,5.648611,7.402917,0.144861,5.172348,20200504
4,2020-05-05,20.429167,7.102778,5.350000,7.886250,0.209861,5.837500,20200505


In [8]:
final_river_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1093 entries, 0 to 1092
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   날짜         1093 non-null   datetime64[ns]
 1   수온         1068 non-null   float64       
 2   pH         1068 non-null   float64       
 3   용존산소(㎎/L)  1068 non-null   float64       
 4   총질소(㎎/L)   1069 non-null   float64       
 5   총인(㎎/L)    1069 non-null   float64       
 6   총유기탄소      1061 non-null   float64       
 7   tm         1093 non-null   object        
dtypes: datetime64[ns](1), float64(6), object(1)
memory usage: 68.4+ KB


In [9]:
print("--- 컬럼별 결측치 합계 ---")
print(final_river_df.isnull().sum())

# 결측치 비율(%) 확인
print("\n--- 결측치 비율 (%) ---")
print((final_river_df.isnull().sum() / len(final_river_df)) * 100)

--- 컬럼별 결측치 합계 ---
날짜            0
수온           25
pH           25
용존산소(㎎/L)    25
총질소(㎎/L)     24
총인(㎎/L)      24
총유기탄소        32
tm            0
dtype: int64

--- 결측치 비율 (%) ---
날짜           0.000000
수온           2.287283
pH           2.287283
용존산소(㎎/L)    2.287283
총질소(㎎/L)     2.195791
총인(㎎/L)      2.195791
총유기탄소        2.927722
tm           0.000000
dtype: float64


In [10]:
# 1. 시계열 흐름에 따라 비어있는 값을 부드럽게 채우기
# limit_direction='both'는 맨 앞이나 맨 뒤가 비어있을 경우까지 고려합니다.
final_river_df_clean = final_river_df.interpolate(method='linear', limit_direction='both')

# 2. 혹시나 남은 결측치가 있다면 (모두 NaN인 컬럼 등) 0이나 평균으로 마무리
final_river_df_clean = final_river_df_clean.fillna(final_river_df_clean.mean(numeric_only=True))

# 최종 확인
print(f"처리 후 결측치 수: {final_river_df_clean.isnull().sum().sum()}")

처리 후 결측치 수: 0


In [12]:
final_river_df.to_csv('data\River_data.csv', index=None, encoding='utf-8')

In [4]:
def analyze_river_file(file_path, river_name):
    # 1. HTML 형식 읽기
    tables = pd.read_html(file_path, encoding='utf-8')
    df = tables[0]
    
    # 2. 날짜 형식 변환 (전처리를 위해 먼저 변환)
    df['날짜'] = pd.to_datetime(df['날짜'].astype(str))
    
    # 3. 5~10월 데이터만 필터링
    df = df[df['날짜'].dt.month.between(5, 10)].copy()
    
    # 4. 분석에 사용할 수치형 컬럼들 (페놀, 시안 등 제외)
    # 실제 파일의 컬럼명과 일치하는지 확인 필요
    numeric_cols = ['수온', 'pH', '용존산소(㎎/L)', '총질소(㎎/L)', '총인(㎎/L)', '총유기탄소']
    
    # 5. 날짜별 평균 계산 (15시 필터링 대신 모든 시간대 평균)
    # numeric_only=True를 넣어 문자열 컬럼(시간 등) 제외
    df_daily = df.groupby('날짜')[numeric_cols].mean(numeric_only=True).reset_index()
    
    # 6. 분석용 컬럼 추가
    df_daily['year'] = df_daily['날짜'].dt.year
    df_daily['month'] = df_daily['날짜'].dt.month
    
    print(f"--- [{river_name}] 일평균 기반 분석 결과 ---")
    print(f"전체 일수(행 개수): {len(df_daily)}")
    print(f"날짜 범위: {df_daily['날짜'].min().date()} ~ {df_daily['날짜'].max().date()}")
    print("연도별 데이터 분포 (일수):")
    print(df_daily['year'].value_counts().sort_index())
    print("-" * 30)
    
    return df_daily

# 2. 개별 파일 실행
df_jung = analyze_river_file('data\Jungryangchun.xls', '중랑천')
df_ahn = analyze_river_file('data\Ahnyangchun.xls', '안양천')
df_tan = analyze_river_file('data\Tanchun.xls', '탄천')

--- [중랑천] 일평균 기반 분석 결과 ---
전체 일수(행 개수): 1093
날짜 범위: 2020-05-01 ~ 2025-10-31
연도별 데이터 분포 (일수):
year
2020    173
2021    184
2022    184
2023    184
2024    184
2025    184
Name: count, dtype: int64
------------------------------
--- [안양천] 일평균 기반 분석 결과 ---
전체 일수(행 개수): 1093
날짜 범위: 2020-05-01 ~ 2025-10-31
연도별 데이터 분포 (일수):
year
2020    173
2021    184
2022    184
2023    184
2024    184
2025    184
Name: count, dtype: int64
------------------------------
--- [탄천] 일평균 기반 분석 결과 ---
전체 일수(행 개수): 1093
날짜 범위: 2020-05-01 ~ 2025-10-31
연도별 데이터 분포 (일수):
year
2020    173
2021    184
2022    184
2023    184
2024    184
2025    184
Name: count, dtype: int64
------------------------------
